In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
def list_files_in_subdirs(parent_dir: str | Path, subdir_names: list[str]) -> list[Path]:
    """
    Given a parent directory and a list of subdirectory names,
    return a list of Path objects for all files inside each subdirectory
    (non-recursive — only direct contents of each subdir).
    """
    parent_dir = Path(parent_dir)
    all_files = []

    for name in subdir_names:
        subdir = parent_dir / name
        if not subdir.is_dir():
            print(f"Warning: {subdir} is not a valid directory, skipping.")
            continue
        all_files.extend(p for p in subdir.iterdir() if p.is_file())

    return all_files


In [6]:
import statistics
from pathlib import Path
import librosa

def audio_length_stats(file_paths: list[str | Path]) -> tuple[float, float]:
    """
    Given a list of audio file paths, return (mean, stdev) of their durations in seconds.
    Uses librosa to compute duration.
    """
    durations = []

    for path in file_paths:
        path = Path(path)
        try:
            duration = librosa.get_duration(path=str(path))
            durations.append(duration)
        except Exception as e:
            print(f"Warning: error reading {path}: {e}, skipping.")

    if not durations:
        raise ValueError("No valid audio files found to compute stats.")

    mean_len = statistics.mean(durations)
    stdev_len = statistics.stdev(durations) if len(durations) > 1 else 0.0

    return mean_len, stdev_len

In [4]:
sub_dir_lst = ["Explosion", "Gunshots", "UAV"]
parent_path = Path(r"..\Data\Audio_Files\Raw_Audios\Anomalies")

audio_files = list_files_in_subdirs(parent_path, sub_dir_lst)

In [7]:
mean, stdev = audio_length_stats(audio_files)

In [ ]:
print(mean)
print(stdev) #in seconds

18.281575907479393
34.67026782856111


In [10]:
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path

def shorten_audio_files(input_dir: str | Path, seg_mean: float = 18.0, seg_stdev: float = 34.0, min_len: float = 10.0):
    """
    For every audio file in input_dir, sample a segment length from
    N(seg_mean, seg_stdev), extract a random segment of that length,
    and save it as .wav in a sibling directory "shortened_unlabeled".
    """
    input_dir = Path(input_dir)
    output_dir = input_dir.parent / "shortened_unlabeled"
    output_dir.mkdir(exist_ok=True)

    audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}

    for file_path in input_dir.iterdir():
        if not file_path.is_file() or file_path.suffix.lower() not in audio_extensions:
            continue

        try:
            full_duration = librosa.get_duration(path=str(file_path))

            # Sample segment length, take abs value, floor at min_len
            seg_len = abs(np.random.normal(seg_mean, seg_stdev))
            seg_len = max(seg_len, min_len)
            seg_len = min(seg_len, full_duration)  # can't exceed file's own length

            max_start = full_duration - seg_len
            start = np.random.uniform(0, max_start) if max_start > 0 else 0.0

            y, sr = librosa.load(str(file_path), sr=None, offset=start, duration=seg_len)

            out_path = output_dir / (file_path.stem + ".wav")
            sf.write(str(out_path), y, sr)

            print(f"{file_path.name}: {seg_len:.2f}s segment @ {start:.2f}s -> {out_path.name}")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    print(f"\nDone. Files saved to: {output_dir}")

In [15]:
f_path = Path(r".\Data_v2\Raw_Audios\Test\Unlabeled")
shorten_audio_files(f_path)

658728__performansas__lithuanian-voice-acting.wav: 17.02s segment @ 0.00s -> 658728__performansas__lithuanian-voice-acting.wav
665234__drdock__approving-voice-male.wav: 3.66s segment @ 0.00s -> 665234__drdock__approving-voice-male.wav
665940__joseviana__amanhecer-na-ilha-com-cachorros-galos-e-barcos-sunrise-on-the-island-with-rooster-dogs-and-boats.wav: 20.38s segment @ 370.65s -> 665940__joseviana__amanhecer-na-ilha-com-cachorros-galos-e-barcos-sunrise-on-the-island-with-rooster-dogs-and-boats.wav
675423__craigsmith__s15-16-horse-trots-in-stable-movement-in-background.wav: 38.19s segment @ 0.20s -> 675423__craigsmith__s15-16-horse-trots-in-stable-movement-in-background.wav
688235__philip_goddard__river-teign-at-night-from-sharp-tor-teign-gorge.flac: 10.77s segment @ 2526.25s -> 688235__philip_goddard__river-teign-at-night-from-sharp-tor-teign-gorge.wav
690720__kevp888__voc_230611-2268_fr_thunderstorm.wav: 10.00s segment @ 791.16s -> 690720__kevp888__voc_230611-2268_fr_thunderstorm.wav

In [ ]:
""" 
Next step is to bootstrap data for final files that will be used for
training testing

Need alot of data for CNN model!
"""

In [2]:
from Models import AudioBootstrapper
from pathlib import Path

bootstrapper = AudioBootstrapper()

og_folder_path = Path(r".\Data_v2\Raw_Audios\Train")
sub_folder_names = ["Airplanes", "Drones", "Explosion", "Gunshots", "Helicopter", "Land_Vehicles", "shortened_unlabeled"] 
output_folder_path = Path(r".\Data_v2\Bootstrapped_Audios\Train") 
overlay_folder_name = "shortened_unlabeled" #overlay augmentation removed internally
aug_technique_num = 4 
num_bootstraps = 700
toc_name = "train_toc"

bootstrapper.BootStrap(og_folder_path, sub_folder_names, output_folder_path, overlay_folder_name, aug_technique_num, num_bootstraps, toc_name)

2026-07-19 18:42:09,884 - INFO - Files processed: 1/4900
2026-07-19 18:42:10,180 - INFO - Files processed: 2/4900
2026-07-19 18:42:10,783 - INFO - Files processed: 3/4900
2026-07-19 18:42:11,088 - INFO - Files processed: 4/4900
2026-07-19 18:42:11,426 - INFO - Files processed: 5/4900
2026-07-19 18:42:11,943 - INFO - Files processed: 6/4900
2026-07-19 18:42:12,076 - INFO - Files processed: 7/4900
2026-07-19 18:42:12,686 - INFO - Files processed: 8/4900
2026-07-19 18:42:12,973 - INFO - Files processed: 9/4900
2026-07-19 18:42:13,459 - INFO - Files processed: 10/4900
2026-07-19 18:42:14,167 - INFO - Files processed: 11/4900
2026-07-19 18:42:14,833 - INFO - Files processed: 12/4900
2026-07-19 18:42:15,382 - INFO - Files processed: 13/4900
2026-07-19 18:42:16,021 - INFO - Files processed: 14/4900
2026-07-19 18:42:16,272 - INFO - Files processed: 15/4900
2026-07-19 18:42:16,569 - INFO - Files processed: 16/4900
2026-07-19 18:42:17,052 - INFO - Files processed: 17/4900
2026-07-19 18:42:17,351

{'file_name': ['audio_1.wav',
  'audio_2.wav',
  'audio_3.wav',
  'audio_4.wav',
  'audio_5.wav',
  'audio_6.wav',
  'audio_7.wav',
  'audio_8.wav',
  'audio_9.wav',
  'audio_10.wav',
  'audio_11.wav',
  'audio_12.wav',
  'audio_13.wav',
  'audio_14.wav',
  'audio_15.wav',
  'audio_16.wav',
  'audio_17.wav',
  'audio_18.wav',
  'audio_19.wav',
  'audio_20.wav',
  'audio_21.wav',
  'audio_22.wav',
  'audio_23.wav',
  'audio_24.wav',
  'audio_25.wav',
  'audio_26.wav',
  'audio_27.wav',
  'audio_28.wav',
  'audio_29.wav',
  'audio_30.wav',
  'audio_31.wav',
  'audio_32.wav',
  'audio_33.wav',
  'audio_34.wav',
  'audio_35.wav',
  'audio_36.wav',
  'audio_37.wav',
  'audio_38.wav',
  'audio_39.wav',
  'audio_40.wav',
  'audio_41.wav',
  'audio_42.wav',
  'audio_43.wav',
  'audio_44.wav',
  'audio_45.wav',
  'audio_46.wav',
  'audio_47.wav',
  'audio_48.wav',
  'audio_49.wav',
  'audio_50.wav',
  'audio_51.wav',
  'audio_52.wav',
  'audio_53.wav',
  'audio_54.wav',
  'audio_55.wav',
  'aud

In [3]:
from Models import AudioBootstrapper
from pathlib import Path

bootstrapper = AudioBootstrapper()

og_folder_path = Path(r".\Data_v2\Raw_Audios\Test")
sub_folder_names = ["Airplanes", "Drones", "Explosion", "Gunshots", "Helicopter", "Land_Vehicles", "shortened_unlabeled"] 
output_folder_path = Path(r".\Data_v2\Bootstrapped_Audios\Test") 
overlay_folder_name = "shortened_unlabeled" #overlay augmentation removed internally
aug_technique_num = 4 
num_bootstraps = 75
toc_name = "test_toc"

bootstrapper.BootStrap(og_folder_path, sub_folder_names, output_folder_path, overlay_folder_name, aug_technique_num, num_bootstraps, toc_name)

2026-07-19 19:05:09,409 - INFO - Files processed: 1/525
2026-07-19 19:05:09,932 - INFO - Files processed: 2/525
2026-07-19 19:05:10,259 - INFO - Files processed: 3/525
2026-07-19 19:05:10,602 - INFO - Files processed: 4/525
2026-07-19 19:05:11,346 - INFO - Files processed: 5/525
2026-07-19 19:05:11,638 - INFO - Files processed: 6/525
2026-07-19 19:05:42,655 - INFO - Files processed: 7/525
2026-07-19 19:05:43,370 - INFO - Files processed: 8/525
2026-07-19 19:05:43,937 - INFO - Files processed: 9/525
2026-07-19 19:05:44,235 - INFO - Files processed: 10/525
2026-07-19 19:05:44,538 - INFO - Files processed: 11/525
2026-07-19 19:05:44,784 - INFO - Files processed: 12/525
2026-07-19 19:06:16,081 - INFO - Files processed: 13/525
2026-07-19 19:06:16,628 - INFO - Files processed: 14/525
2026-07-19 19:06:16,921 - INFO - Files processed: 15/525
2026-07-19 19:06:17,213 - INFO - Files processed: 16/525
2026-07-19 19:06:17,950 - INFO - Files processed: 17/525
2026-07-19 19:06:18,531 - INFO - Files p

{'file_name': ['audio_1.wav',
  'audio_2.wav',
  'audio_3.wav',
  'audio_4.wav',
  'audio_5.wav',
  'audio_6.wav',
  'audio_7.wav',
  'audio_8.wav',
  'audio_9.wav',
  'audio_10.wav',
  'audio_11.wav',
  'audio_12.wav',
  'audio_13.wav',
  'audio_14.wav',
  'audio_15.wav',
  'audio_16.wav',
  'audio_17.wav',
  'audio_18.wav',
  'audio_19.wav',
  'audio_20.wav',
  'audio_21.wav',
  'audio_22.wav',
  'audio_23.wav',
  'audio_24.wav',
  'audio_25.wav',
  'audio_26.wav',
  'audio_27.wav',
  'audio_28.wav',
  'audio_29.wav',
  'audio_30.wav',
  'audio_31.wav',
  'audio_32.wav',
  'audio_33.wav',
  'audio_34.wav',
  'audio_35.wav',
  'audio_36.wav',
  'audio_37.wav',
  'audio_38.wav',
  'audio_39.wav',
  'audio_40.wav',
  'audio_41.wav',
  'audio_42.wav',
  'audio_43.wav',
  'audio_44.wav',
  'audio_45.wav',
  'audio_46.wav',
  'audio_47.wav',
  'audio_48.wav',
  'audio_49.wav',
  'audio_50.wav',
  'audio_51.wav',
  'audio_52.wav',
  'audio_53.wav',
  'audio_54.wav',
  'audio_55.wav',
  'aud

In [ ]:
""" 
Note to prevent data leakage source audio were seperated into train test at the start
bootstrapped train test audios were constructed from seperate source files
"""

Code to Shorten Helicopter and Land Vehicle Audios

In [9]:
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path

def shorten_audio_files(input_dir: str | Path, output_name, n: float = 18.0):
    """
    For every audio file in input_dir, extract a random n-second segment
    (or the entire clip if it's shorter than n seconds), and save it as
    .wav in a sibling directory "shortened_{output_name}".
    """
    input_dir = Path(input_dir)
    output_dir = input_dir.parent / f"shortened_{output_name}"
    output_dir.mkdir(exist_ok=True)

    audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}

    for file_path in input_dir.iterdir():
        if not file_path.is_file() or file_path.suffix.lower() not in audio_extensions:
            continue

        try:
            full_duration = librosa.get_duration(path=str(file_path))

            if full_duration <= n:
                # Clip is shorter than (or equal to) n seconds -> keep it whole
                seg_len = full_duration
                start = 0.0
            else:
                seg_len = n
                max_start = full_duration - seg_len
                start = np.random.uniform(0, max_start)

            y, sr = librosa.load(str(file_path), sr=None, offset=start, duration=seg_len)

            out_path = output_dir / (file_path.stem + ".wav")
            sf.write(str(out_path), y, sr)

            print(f"{file_path.name}: {seg_len:.2f}s segment @ {start:.2f}s -> {out_path.name}")

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    print(f"\nDone. Files saved to: {output_dir}")


In [11]:
input_dir = r"..\Raw_Audios\Anomalies\Land_Vehicles"
output_name = "Land_Vehicles"

shorten_audio_files(input_dir, output_name, 60)

107037__cameron-johnson__pegasus-starting.wav: 34.76s segment @ 0.00s -> 107037__cameron-johnson__pegasus-starting.wav
161879__joniheinonen__war-heavy-vehicles-passing-by-close-and-distant-passing-vehicles-people-talkin-finnish-wind-0001.wav: 60.00s segment @ 5.93s -> 161879__joniheinonen__war-heavy-vehicles-passing-by-close-and-distant-passing-vehicles-people-talkin-finnish-wind-0001.wav
16768__dobroide__20060308brakes.wav: 7.41s segment @ 0.00s -> 16768__dobroide__20060308brakes.wav
182793__qubodup__military-vehicle-humming.flac: 30.00s segment @ 0.00s -> 182793__qubodup__military-vehicle-humming.wav
187564__qubodup__truck-engine-idle-loops.flac: 38.02s segment @ 0.00s -> 187564__qubodup__truck-engine-idle-loops.wav
257034__jagadamba__vehicle-passing-driving-car-lorry-van-bus-tractor-truck.wav: 4.58s segment @ 0.00s -> 257034__jagadamba__vehicle-passing-driving-car-lorry-van-bus-tractor-truck.wav
262759__temawas__car-pass.wav: 7.76s segment @ 0.00s -> 262759__temawas__car-pass.wav
32

Bootstrapping Data_v3

In [2]:
from Models import AudioBootstrapper
from pathlib import Path

bootstrapper = AudioBootstrapper()

og_folder_path = Path(r".\Data_v3\Raw_Audios\Train")
sub_folder_names = ["Airplanes", "Drones", "Explosion", "Gunshots", "Helicopter", "Land_Vehicles", "shortened_unlabeled"] 
output_folder_path = Path(r".\Data_v3\Bootstrapped_Audios\Train") 
aug_technique_num = 5 
num_bootstraps = 200 #200 each class train, 75 each class test 
toc_name = "train_toc"
bootstrapper.BootStrap_v2(og_folder_path, sub_folder_names, output_folder_path, aug_technique_num, num_bootstraps, toc_name)

2026-07-21 17:40:25,333 - INFO - Files processed: 1/1400
2026-07-21 17:40:25,983 - INFO - Files processed: 2/1400
2026-07-21 17:40:26,254 - INFO - Files processed: 3/1400
2026-07-21 17:40:27,582 - INFO - Files processed: 4/1400
2026-07-21 17:40:27,933 - INFO - Files processed: 5/1400
2026-07-21 17:40:28,610 - INFO - Files processed: 6/1400
2026-07-21 17:40:29,080 - INFO - Files processed: 7/1400
2026-07-21 17:40:30,088 - INFO - Files processed: 8/1400
2026-07-21 17:40:30,801 - INFO - Files processed: 9/1400
2026-07-21 17:40:31,278 - INFO - Files processed: 10/1400
2026-07-21 17:40:31,942 - INFO - Files processed: 11/1400
2026-07-21 17:40:32,429 - INFO - Files processed: 12/1400
2026-07-21 17:40:32,516 - WARNING - overlay_skipped_no_candidates_or_zero_chosen
2026-07-21 17:40:33,132 - INFO - Files processed: 13/1400
2026-07-21 17:40:33,955 - INFO - Files processed: 14/1400
2026-07-21 17:40:34,236 - INFO - Files processed: 15/1400
2026-07-21 17:40:34,912 - INFO - Files processed: 16/1400


{'file_name': ['audio_1.wav',
  'audio_2.wav',
  'audio_3.wav',
  'audio_4.wav',
  'audio_5.wav',
  'audio_6.wav',
  'audio_7.wav',
  'audio_8.wav',
  'audio_9.wav',
  'audio_10.wav',
  'audio_11.wav',
  'audio_12.wav',
  'audio_13.wav',
  'audio_14.wav',
  'audio_15.wav',
  'audio_16.wav',
  'audio_17.wav',
  'audio_18.wav',
  'audio_19.wav',
  'audio_20.wav',
  'audio_21.wav',
  'audio_22.wav',
  'audio_23.wav',
  'audio_24.wav',
  'audio_25.wav',
  'audio_26.wav',
  'audio_27.wav',
  'audio_28.wav',
  'audio_29.wav',
  'audio_30.wav',
  'audio_31.wav',
  'audio_32.wav',
  'audio_33.wav',
  'audio_34.wav',
  'audio_35.wav',
  'audio_36.wav',
  'audio_37.wav',
  'audio_38.wav',
  'audio_39.wav',
  'audio_40.wav',
  'audio_41.wav',
  'audio_42.wav',
  'audio_43.wav',
  'audio_44.wav',
  'audio_45.wav',
  'audio_46.wav',
  'audio_47.wav',
  'audio_48.wav',
  'audio_49.wav',
  'audio_50.wav',
  'audio_51.wav',
  'audio_52.wav',
  'audio_53.wav',
  'audio_54.wav',
  'audio_55.wav',
  'aud

In [3]:
from Models import AudioBootstrapper
from pathlib import Path

bootstrapper = AudioBootstrapper()

og_folder_path = Path(r".\Data_v3\Raw_Audios\Test")
sub_folder_names = ["Airplanes", "Drones", "Explosion", "Gunshots", "Helicopter", "Land_Vehicles", "shortened_unlabeled"] 
output_folder_path = Path(r".\Data_v3\Bootstrapped_Audios\Test") 
aug_technique_num = 5 
num_bootstraps = 75 

toc_name = "test_toc"

bootstrapper.BootStrap_v2(og_folder_path, sub_folder_names, output_folder_path, aug_technique_num, num_bootstraps, toc_name)

2026-07-21 17:52:26,430 - INFO - Files processed: 1/525
2026-07-21 17:52:27,158 - INFO - Files processed: 2/525
2026-07-21 17:52:27,794 - INFO - Files processed: 3/525
2026-07-21 17:52:28,195 - INFO - Files processed: 4/525
2026-07-21 17:52:28,908 - INFO - Files processed: 5/525
2026-07-21 17:52:29,649 - INFO - Files processed: 6/525
2026-07-21 17:52:30,544 - INFO - Files processed: 7/525
2026-07-21 17:52:31,109 - INFO - Files processed: 8/525
2026-07-21 17:52:31,664 - INFO - Files processed: 9/525
2026-07-21 17:52:32,330 - WARNING - overlay_skipped_no_candidates_or_zero_chosen
2026-07-21 17:52:32,330 - INFO - Files processed: 10/525
2026-07-21 17:52:32,471 - INFO - Files processed: 11/525
2026-07-21 17:52:33,107 - INFO - Files processed: 12/525
2026-07-21 17:52:33,882 - INFO - Files processed: 13/525
2026-07-21 17:52:34,704 - INFO - Files processed: 14/525
2026-07-21 17:52:35,375 - INFO - Files processed: 15/525
2026-07-21 17:52:36,128 - INFO - Files processed: 16/525
2026-07-21 17:52

{'file_name': ['audio_1.wav',
  'audio_2.wav',
  'audio_3.wav',
  'audio_4.wav',
  'audio_5.wav',
  'audio_6.wav',
  'audio_7.wav',
  'audio_8.wav',
  'audio_9.wav',
  'audio_10.wav',
  'audio_11.wav',
  'audio_12.wav',
  'audio_13.wav',
  'audio_14.wav',
  'audio_15.wav',
  'audio_16.wav',
  'audio_17.wav',
  'audio_18.wav',
  'audio_19.wav',
  'audio_20.wav',
  'audio_21.wav',
  'audio_22.wav',
  'audio_23.wav',
  'audio_24.wav',
  'audio_25.wav',
  'audio_26.wav',
  'audio_27.wav',
  'audio_28.wav',
  'audio_29.wav',
  'audio_30.wav',
  'audio_31.wav',
  'audio_32.wav',
  'audio_33.wav',
  'audio_34.wav',
  'audio_35.wav',
  'audio_36.wav',
  'audio_37.wav',
  'audio_38.wav',
  'audio_39.wav',
  'audio_40.wav',
  'audio_41.wav',
  'audio_42.wav',
  'audio_43.wav',
  'audio_44.wav',
  'audio_45.wav',
  'audio_46.wav',
  'audio_47.wav',
  'audio_48.wav',
  'audio_49.wav',
  'audio_50.wav',
  'audio_51.wav',
  'audio_52.wav',
  'audio_53.wav',
  'audio_54.wav',
  'audio_55.wav',
  'aud

In [3]:
import json
import pandas as pd
j_path = r".\Data_v3\Bootstrapped_Audios\train_toc.json"
with open(Path(j_path), "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df[:30]


,file_name,base_audio,overlay_Airplanes,overlay_Drones,overlay_Explosion,overlay_Gunshots,overlay_Helicopter,overlay_Land_Vehicles,overlay_shortened_unlabeled
0,audio_1.wav,Airplanes,0,0,0,0,0,0,0
1,audio_2.wav,Airplanes,0,0,0,0,0,0,0
2,audio_3.wav,Airplanes,0,0,0,0,0,0,0
3,audio_4.wav,Airplanes,1,1,0,1,0,1,1
4,audio_5.wav,Airplanes,0,1,1,1,1,1,1
5,audio_6.wav,Airplanes,0,0,0,0,0,0,0
6,audio_7.wav,Airplanes,1,0,0,0,0,0,0
7,audio_8.wav,Airplanes,1,1,0,1,1,0,1
8,audio_9.wav,Airplanes,0,0,1,1,1,1,1
9,audio_10.wav,Airplanes,0,0,0,0,0,0,0


In [1]:
import os
import librosa
import soundfile as sf
from pathlib import Path

def truncate_audios(folder_path, output_folder=None, max_duration=60.0, extensions=(".wav", ".mp3", ".flac", ".ogg", ".m4a")):
    """
    Walks through folder_path, loads each audio file, and truncates it to
    max_duration seconds (default 60s / 1 min). Saves results either back
    in place or into output_folder (mirroring filenames).

    Parameters
    ----------
    folder_path : str
        Path to the folder containing audio files.
    output_folder : str or None
        Where to save truncated files. If None, overwrites original files.
    max_duration : float
        Max length in seconds (default 60.0).
    extensions : tuple
        File extensions to process.

    Returns
    -------
    list of dicts with filename, original_duration, new_duration
    """
    if output_folder is not None:
        os.makedirs(output_folder, exist_ok=True)

    results = []

    for fname in os.listdir(folder_path):
        if not fname.lower().endswith(extensions):
            continue

        in_path = os.path.join(folder_path, fname)

        try:
            # sr=None preserves original sample rate
            y, sr = librosa.load(in_path, sr=None, mono=False)
        except Exception as e:
            print(f"Skipping {fname}: could not load ({e})")
            continue

        # librosa gives (n,) for mono or (channels, n) for multi-channel
        n_samples = y.shape[-1]
        orig_duration = n_samples / sr
        max_samples = int(max_duration * sr)

        if n_samples > max_samples:
            y_trunc = y[..., :max_samples] if y.ndim > 1 else y[:max_samples]
        else:
            y_trunc = y

        new_duration = y_trunc.shape[-1] / sr

        out_path = os.path.join(output_folder, fname) if output_folder else in_path

        # soundfile expects (samples, channels) for multi-channel audio
        data_to_write = y_trunc.T if y_trunc.ndim > 1 else y_trunc
        sf.write(out_path, data_to_write, sr)

        results.append({
            "filename": fname,
            "original_duration_sec": round(orig_duration, 2),
            "new_duration_sec": round(new_duration, 2),
        })

        print(f"{fname}: {orig_duration:.2f}s -> {new_duration:.2f}s")

    return results


if __name__ == "__main__":
    # Example usage
    truncate_audios(
        folder_path=Path(r".\Data_v3\Bootstrapped_Audios\Test"),
        output_folder=None,  # set to None to overwrite in place
        max_duration=120.0,
    )

audio_1.wav: 60.00s -> 60.00s
audio_10.wav: 47.20s -> 47.20s
audio_100.wav: 47.20s -> 47.20s
audio_1000.wav: 60.00s -> 60.00s
audio_1001.wav: 60.00s -> 60.00s
audio_1002.wav: 60.00s -> 60.00s
audio_1003.wav: 56.45s -> 56.45s
audio_1004.wav: 60.00s -> 60.00s
audio_1005.wav: 55.57s -> 55.57s
audio_1006.wav: 60.00s -> 60.00s
audio_1007.wav: 58.75s -> 58.75s
audio_1008.wav: 56.24s -> 56.24s
audio_1009.wav: 60.00s -> 60.00s
audio_101.wav: 42.83s -> 42.83s
audio_1010.wav: 60.00s -> 60.00s
audio_1011.wav: 60.00s -> 60.00s
audio_1012.wav: 53.21s -> 53.21s
audio_1013.wav: 60.00s -> 60.00s
audio_1014.wav: 60.00s -> 60.00s
audio_1015.wav: 58.62s -> 58.62s
audio_1016.wav: 60.00s -> 60.00s
audio_1017.wav: 60.00s -> 60.00s
audio_1018.wav: 55.77s -> 55.77s
audio_1019.wav: 60.00s -> 60.00s
audio_102.wav: 37.44s -> 37.44s
audio_1020.wav: 59.59s -> 59.59s
audio_1021.wav: 60.00s -> 60.00s
audio_1022.wav: 56.14s -> 56.14s
audio_1023.wav: 60.00s -> 60.00s
audio_1024.wav: 49.14s -> 49.14s
audio_1025.wav: 54